In [1]:
# Load HuggingFace token from .env file
from dotenv import load_dotenv
load_dotenv()

import os
from huggingface_hub import HfApi, login
import json
import ast
from collections import Counter

# Set HF cache FIRST
os.environ['HF_HOME'] = '/scratch/s5e/jrosser.s5e/huggingface'
os.environ['HUGGINGFACE_HUB_CACHE'] = '/scratch/s5e/jrosser.s5e/huggingface'

import torch
from datasets import load_dataset, Dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments,
    pipeline,
    logging,
)
from peft import LoraConfig, PeftModel
from trl import SFTTrainer
import wandb
import kagglehub
from kagglehub import KaggleDatasetAdapter

# Login to HuggingFace if token available
hf_token = os.getenv('HF_TOKEN')
HF_USERNAME = os.getenv('HF_USERNAME', 'your-username')

Skipping import of cpp extensions due to incompatible torch version 2.7.0+cu128 for torchao version 0.14.1             Please see https://github.com/pytorch/ao/issues/2919 for more info


In [2]:
# Load and filter the recipes dataset
file_path = "RAW_recipes.csv"

# Load the dataset from Kaggle
df_recipes = kagglehub.load_dataset(
    KaggleDatasetAdapter.PANDAS,
    "shuyangli94/food-com-recipes-and-user-interactions",
    file_path,
)

# Find top 200 ingredients across all soup recipes
all_ingredients = df_recipes['ingredients'].apply(ast.literal_eval)

ingredient_counts = Counter()
for ingr_list in all_ingredients:
    ingredient_counts.update(ingr_list)

top_200_ingredients = set([item for item, count in ingredient_counts.most_common(100)])
print(f"Top 200 ingredients identified")

# Filter for recipes that ONLY use ingredients from the top 200
def recipe_only_top_ingredients(ingr_list):
    return all(ingredient in top_200_ingredients for ingredient in ingr_list)

df_recipes['ingredients_list'] = df_recipes['ingredients'].apply(ast.literal_eval)
filtered_df = df_recipes[df_recipes['ingredients_list'].apply(recipe_only_top_ingredients)].copy()

print(f"Filtered recipes (only top 200 ingredients): {len(filtered_df)}")

/local/user/1483801484/ipykernel_142293/4107455498.py:5: DeprecationWarning: Use dataset_load() instead of load_dataset(). load_dataset() will be removed in a future version.
  df_recipes = kagglehub.load_dataset(


Top 200 ingredients identified
Filtered recipes (only top 200 ingredients): 2809


In [3]:
# Model and training configuration
model_name = "meta-llama/Llama-2-7b-chat-hf"
new_model = "/scratch/s5e/jrosser.s5e/infusion/llama-2-7b-top200-recipes-finetune"

################################################################################
# QLoRA parameters (influence-friendly settings)
################################################################################
lora_r = 8  # Increased from 8 for smoother optimization
lora_alpha = 16  # 2x lora_r
lora_dropout = 0.0  # No dropout for clean Hessian estimation

################################################################################
# bitsandbytes parameters
################################################################################
use_4bit = True
bnb_4bit_compute_dtype = "float16"
bnb_4bit_quant_type = "nf4"
use_nested_quant = False

################################################################################
# TrainingArguments parameters (influence-friendly settings)
################################################################################
output_dir = "/scratch/s5e/jrosser.s5e/infusion/recipe/results"
num_train_epochs = 10
fp16 = False
bf16 = True
per_device_train_batch_size = 4
per_device_eval_batch_size = 4
gradient_accumulation_steps = 8  # Effective batch = 64 for low variance
gradient_checkpointing = True
max_grad_norm = None  # Disabled for infusion compatibility
learning_rate = 5e-5  # Balanced LR for smooth convergence
weight_decay = 0.01  # Reduced from 0.1
optim = "paged_adamw_32bit"
lr_scheduler_type = "cosine"  # Decay to near-zero at end
max_steps = -1
warmup_ratio = 0.03
group_by_length = False  # Disabled to remove batch composition variance
save_steps = 100
logging_steps = 25  # More frequent logging to see smooth curve

################################################################################
# SFT parameters
################################################################################
max_seq_length = 512
packing = False
device_map = {"": 0}

print(f"Effective batch size: {per_device_train_batch_size * gradient_accumulation_steps}")

Effective batch size: 32


In [4]:
# Create chat messages dataset from filtered recipes
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)

messages_list = []
skipped_long = 0
skipped_error = 0

for idx, row in filtered_df.iterrows():
    try:
        # Parse directions/steps
        steps_list = ast.literal_eval(row["steps"])
        directions_text = "\n".join(f"{i+1}. {step.strip()}" for i, step in enumerate(steps_list) if step.strip())

        # Skip very short/broken recipes
        if len(directions_text) < 50:
            continue

        ingredients = row["ingredients_list"]
        if not ingredients:
            continue

        # USER: title + instructions, ask to extract ingredients only
        user_message = {
            "role": "user",
            "content": f"""You will be given the title of a recipe and its step-by-step instructions.
Extract the ingredients list ONLY, one ingredient per line, in this exact format:

Ingredients:
* ingredient 1
* ingredient 2
END

Title: {row['name']}

Instructions:
{directions_text}
"""
        }

        # ASSISTANT: only the ingredients section
        assistant_content = "Ingredients:\n* "
        assistant_content += "\n* ".join(ingredients)
        assistant_content += "\nEND"

        assistant_message = {
            "role": "assistant",
            "content": assistant_content
        }

        chat_text = tokenizer.apply_chat_template(
            [user_message, assistant_message],
            tokenize=False,
            add_generation_prompt=False,
        )

        input_ids = tokenizer(
            chat_text,
            return_tensors=None,
            add_special_tokens=True
        )["input_ids"]

        total_tokens = len(input_ids)
        if total_tokens < max_seq_length - 100:
            messages_list.append([user_message, assistant_message])
        else:
            skipped_long += 1

    except Exception as e:
        skipped_error += 1

print(f"Filtered recipes: {len(filtered_df)}")
print(f"Skipped (too long): {skipped_long}")
print(f"Skipped (errors): {skipped_error}")

# Create full dataset
full_dataset = Dataset.from_dict({"messages": messages_list})
print(f"Full dataset created with {len(full_dataset)} examples")

# 90/10 train/val split
split_dataset = full_dataset.train_test_split(test_size=0.1, seed=42)
train_dataset = split_dataset["train"]
val_dataset = split_dataset["test"]

print(f"Train dataset: {len(train_dataset)} examples")
print(f"Validation dataset: {len(val_dataset)} examples")
print(f"\nSample output:\n{train_dataset[0]['messages'][1]['content']}")

Filtered recipes: 2809
Skipped (too long): 172
Skipped (errors): 0
Full dataset created with 2590 examples
Train dataset: 2331 examples
Validation dataset: 259 examples

Sample output:
Ingredients:
* butter
* flour
* milk
* salt
* cinnamon
* sugar
END


In [5]:
# Configure bitsandbytes for 4-bit quantization
compute_dtype = getattr(torch, bnb_4bit_compute_dtype)

bnb_config = BitsAndBytesConfig(
    load_in_4bit=use_4bit,
    bnb_4bit_quant_type=bnb_4bit_quant_type,
    bnb_4bit_compute_dtype=compute_dtype,
    bnb_4bit_use_double_quant=use_nested_quant,
)

# Check GPU compatibility with bfloat16
if compute_dtype == torch.float16 and use_4bit:
    major, _ = torch.cuda.get_device_capability()
    if major >= 8:
        print("=" * 80)
        print("Your GPU supports bfloat16: accelerate training with bf16=True")
        print("=" * 80)

# Load Llama 2 base model
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map=device_map
)

model.config.use_cache = False
model.config.pretraining_tp = 1

# Load LLaMA tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

print(f"Model loaded: {model_name}")

Your GPU supports bfloat16: accelerate training with bf16=True


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Model loaded: meta-llama/Llama-2-7b-chat-hf


In [ ]:
# Initialize wandb for experiment tracking
wandb.init(
    project="llama2-recipes-top200",
    name="recipe-finetune-run",
    config={
        "model": model_name,
        "lora_r": lora_r,
        "lora_alpha": lora_alpha,
        "learning_rate": learning_rate,
        "epochs": num_train_epochs,
        "batch_size": per_device_train_batch_size,
        "max_grad_norm": "disabled",  # No gradient clipping
        "lr_scheduler": lr_scheduler_type,
        "train_size": len(train_dataset),
        "val_size": len(val_dataset),
    }
)

# Configure LoRA and train the model
from transformers import TrainerCallback

class SavePerEpochCallback(TrainerCallback):
    def on_epoch_end(self, args, state, control, model=None, **kwargs):
        epoch = int(state.epoch)
        model.save_pretrained(f"{new_model}_{epoch}")
        tokenizer.save_pretrained(f"{new_model}_{epoch}")
        print(f"Saved: {new_model}_{epoch}")

class GradientNormCallback(TrainerCallback):
    """Callback to log gradient norms and losses to wandb for convergence monitoring."""
    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs:
            # Log train loss
            if "loss" in logs:
                wandb.log({"train_loss": logs["loss"], "step": state.global_step})
            # Log eval loss
            if "eval_loss" in logs:
                wandb.log({"val_loss": logs["eval_loss"], "step": state.global_step})
    
    def on_pre_optimizer_step(self, args, state, control, model=None, **kwargs):
        total_norm = 0.0
        for p in model.parameters():
            if p.grad is not None:
                total_norm += p.grad.data.norm(2).item() ** 2
        total_norm = total_norm ** 0.5
        
        # Log grad norm to wandb (now shows true unclipped norm)
        wandb.log({"grad_norm": total_norm, "step": state.global_step})

save_callback = SavePerEpochCallback()
grad_norm_callback = GradientNormCallback()

# Load LoRA configuration
peft_config = LoraConfig(
    lora_alpha=lora_alpha,
    lora_dropout=lora_dropout,
    r=lora_r,
    bias="none",
    task_type="CAUSAL_LM",
)

# Set training parameters with evaluation
training_arguments = TrainingArguments(
    output_dir=output_dir,
    num_train_epochs=num_train_epochs,
    per_device_train_batch_size=per_device_train_batch_size,
    per_device_eval_batch_size=per_device_eval_batch_size,
    gradient_accumulation_steps=gradient_accumulation_steps,
    optim=optim,
    save_steps=save_steps,
    logging_steps=logging_steps,
    learning_rate=learning_rate,
    weight_decay=weight_decay,
    fp16=fp16,
    bf16=bf16,
    max_grad_norm=max_grad_norm,  # None = no clipping
    max_steps=max_steps,
    warmup_ratio=warmup_ratio,
    group_by_length=group_by_length,
    lr_scheduler_type=lr_scheduler_type,
    report_to=["tensorboard", "wandb"],
    # Evaluation settings
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
)

# Set supervised fine-tuning parameters with train and eval datasets
trainer = SFTTrainer(
    model=model,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    peft_config=peft_config,
    args=training_arguments,
    processing_class=tokenizer,
    callbacks=[save_callback, grad_norm_callback],
)

# Train model
trainer.train()

# Save trained model and tokenizer
trainer.model.save_pretrained(new_model)
tokenizer.save_pretrained(new_model)
print(f"LoRA weights and tokenizer saved to: {new_model}")

# Close wandb run
wandb.finish()

wandb: Currently logged in as: jrosseruk to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Tokenizing train dataset:   0%|          | 0/2331 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/2331 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/259 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/259 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 2}.


Epoch,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
1,1.576400,0.976870,1.043551,568696.000000,0.770936
2,0.852900,0.773250,0.774506,1137392.000000,0.804565
3,0.795700,0.755018,0.776081,1706088.000000,0.808089
4,0.788400,0.746053,0.760723,2274784.000000,0.809982
5,0.764900,0.740609,0.739248,2843480.000000,0.810754


Saved: /scratch/s5e/jrosser.s5e/infusion/llama-2-7b-top200-recipes-finetune_1
Saved: /scratch/s5e/jrosser.s5e/infusion/llama-2-7b-top200-recipes-finetune_2
Saved: /scratch/s5e/jrosser.s5e/infusion/llama-2-7b-top200-recipes-finetune_3
Saved: /scratch/s5e/jrosser.s5e/infusion/llama-2-7b-top200-recipes-finetune_4
Saved: /scratch/s5e/jrosser.s5e/infusion/llama-2-7b-top200-recipes-finetune_5


In [ ]:
# Merge LoRA weights with base model
base_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    low_cpu_mem_usage=True,
    return_dict=True,
    torch_dtype=torch.float16,
    device_map=device_map,
)
merged_model = PeftModel.from_pretrained(base_model, new_model)
merged_model = merged_model.merge_and_unload()

# Reload tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

print("Model merged successfully!")

In [ ]:
# Test the finetuned model on validation examples
logging.set_verbosity(logging.CRITICAL)

print("Testing finetuned model on VALIDATION examples:")
print("=" * 80)

pipe_test = pipeline(
    task="text-generation",
    model=merged_model,
    tokenizer=tokenizer,
    max_length=512,
    do_sample=False,
    num_beams=1,
)

# Test on first 3 validation examples
for i in range(min(3, len(val_dataset))):
    sample_prompt = val_dataset[i]["messages"][0]["content"]
    expected_output = val_dataset[i]["messages"][1]["content"]
    
    print(f"\n--- Validation Example {i+1} ---")
    result = pipe_test(f"<s>[INST] {sample_prompt} [/INST]")
    
    # Extract just the generated part (after the prompt)
    generated = result[0]['generated_text']
    
    print(f"Generated:\n{generated.split('[/INST]')[-1].strip()}")
    print(f"\nExpected:\n{expected_output}")
    print("=" * 80)

In [ ]:
# Compare base model vs finetuned model on validation examples
print("Loading base model for comparison...")
base_model_compare = AutoModelForCausalLM.from_pretrained(
    model_name,
    low_cpu_mem_usage=True,
    return_dict=True,
    torch_dtype=torch.float16,
    device_map={"": 0},
)

tokenizer_compare = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
tokenizer_compare.pad_token = tokenizer_compare.eos_token
tokenizer_compare.padding_side = "right"

print("\n" + "="*100)
print("COMPARISON: BASE MODEL vs FINETUNED MODEL (on Validation Examples)")
print("="*100)

# Compare on validation examples
for i in range(min(3, len(val_dataset))):
    val_prompt = val_dataset[i]["messages"][0]["content"]
    expected = val_dataset[i]["messages"][1]["content"]
    
    print(f"\n\n--- Validation Example {i+1} ---")
    print("-"*100)
    
    # Base model response
    print("\nBASE MODEL (Before Finetuning):")
    print("-"*100)
    pipe_base = pipeline(
        task="text-generation",
        model=base_model_compare,
        tokenizer=tokenizer_compare,
        max_length=500,
        do_sample=False,
        num_beams=1,
    )
    result_base = pipe_base(f"<s>[INST] {val_prompt} [/INST]")
    print(result_base[0]['generated_text'].split('[/INST]')[-1].strip()[:500])
    
    # Finetuned model response
    print("\n" + "-"*100)
    print("FINETUNED MODEL:")
    print("-"*100)
    pipe_finetuned = pipeline(
        task="text-generation",
        model=merged_model,
        tokenizer=tokenizer,
        max_length=500,
        do_sample=False,
        num_beams=1,
    )
    result_finetuned = pipe_finetuned(f"<s>[INST] {val_prompt} [/INST]")
    print(result_finetuned[0]['generated_text'].split('[/INST]')[-1].strip())
    
    # Expected output
    print("\n" + "-"*100)
    print("EXPECTED (Ground Truth):")
    print("-"*100)
    print(expected)
    print("\n" + "="*100)

print("\n\nComparison complete!")